# Setup

This section defines classes for expressions.

Two main differences from the lecture notes:

- The index `i` of the input variables `x_i` starts from 0 instead of 1.
- The if-then-else case is not considered here. Rather the sqrt case is added to the definition.

In [ ]:
import typing, math, fractions

In [ ]:
class Expr:
    def __init__(self): raise NotImplementedError
    def __str__ (self): raise NotImplementedError

class Const(Expr): # c
    def __init__(self, c: float):
        assert(isinstance(c, float))
        self.c = c

    def __str__(self):
        return f"{self.c:.2e}"

class Var(Expr): # x_i
    def __init__(self, i: int):
        assert(isinstance(i, int))
        self.i = i

    def __str__(self):
        return f"x{self.i}"

class Neg(Expr): # -e1
    def __init__(self, e1: Expr):
        assert(isinstance(e1, Expr))
        self.e1 = e1

    def __str__(self):
        return f"(-{self.e1})"

class Add(Expr): # e1 + e2
    def __init__(self, e1: Expr, e2: Expr):
        assert(isinstance(e1, Expr))
        assert(isinstance(e2, Expr))
        self.e1 = e1
        self.e2 = e2

    def __str__(self):
        return f"({self.e1}+{self.e2})"

class Mul(Expr): # e1 * e2
    def __init__(self, e1: Expr, e2: Expr):
        assert(isinstance(e1, Expr))
        assert(isinstance(e2, Expr))
        self.e1 = e1
        self.e2 = e2

    def __str__(self):
        return f"({self.e1}*{self.e2})"

class Sqrt(Expr): # sqrt(e1)
    def __init__(self, e1: Expr):
        assert(isinstance(e1, Expr))
        self.e1 = e1

    def __str__(self):
        return f"(sqrt{self.e1})"

# Problem 6(c)

Implement the functions `get_range()`, `get_error()`, `make_poly1()`, and `make_poly2()` below, and complete the task at the end.
The functions `range` and `error` are defined in Problem 6 of `hw2.pdf`.

Assumptions (for all functions below).

- No overflow occurs in the evaluation of `e`.
- The argument of every `sqrt` operation always evaluates to a positive value.

## `get_range()`

- It takes an expression `e` and a list `intvs`.
  - The `i`th entry of `intvs` represents the range of the input variable `x_i`.
  - In particular, `intvs[i]` is a tuple of two floats `(l_i, r_i)`, denoting the interval $[l_i, r_i] \cap \mathbb{F}$.
- It returns a tuple of two floats `(l, r)` such that:
  - $[[e]] \sigma \in [l, r] \cap \mathbb{F}$ for all $\sigma \in ([l_0, r_0] \cap \mathbb{F}) \times \cdots \times ([l_{n-1}, r_{n-1}] \cap \mathbb{F})$,
  - where $[[e]]$ is the semantics for `float` (i.e., 64-bit double-precision format).

Notes.

- The output must be a **tuple of two floats**.
- For computing the square root of a float, use `sqrt_float()`.

In [ ]:
def sqrt_float(x: float) -> float:
    return math.sqrt(x)

def get_range(e: Expr, intvs: list[tuple[float, float]]) -> tuple[float, float]:
    assert(isinstance(e, Expr))
    assert(isinstance(intvs, list))
    assert(all( isinstance(intv[0], float) and
                isinstance(intv[1], float) for intv in intvs ))

    if isinstance(e, Const):
        c = e.c
        (l, r) = (c, c)
        return (l, r)

    if isinstance(e, Var):
        i = e.i
        (l, r) = intvs[i]
        return (l, r)

    if isinstance(e, Neg):
        (l1, r1) = get_range(e.e1, intvs)
        (l, r) = (-r1, -l1)
        return (l, r)

    if isinstance(e, Add):
        l1, r1 = get_range(e.e1, intvs)
        l2, r2 = get_range(e.e2, intvs)
        return (l1 + l2, r1 + r2)

    if isinstance(e, Mul):
        l1, r1 = get_range(e.e1, intvs)
        l2, r2 = get_range(e.e2, intvs)
        corners = (l1 * l2, l1 * r2, r1 * l2, r1 * r2)
        return (min(corners), max(corners))

    if isinstance(e, Sqrt):
        l1, r1 = get_range(e.e1, intvs)
        # Reachable sqrt inputs are nonnegative by the problem's promise.
        assert r1 >= 0.0
        return (sqrt_float(max(0.0, l1)), sqrt_float(r1))


In [ ]:
# Simple tests
intvs = [(-1.0, 2.0), (3.0, 4.0)]
e1 = Add( Neg(Var(1)), Const(5.0) ) # -x1 + 5.0
e2 = Mul( Var(0), Var(0) )          # x0 * x0
print( f"intvs = {intvs}" )
print( f"range({e1}) = {get_range(e1, intvs)}" )
print( f"range({e2}) = {get_range(e2, intvs)}" )

## `get_error()`

- It takes an expression `e` and a list `intvs` as in `get_range()`.
- It returns a rational number `err` such that:
  - $\Big| [[e]] \sigma - [[e]]_r \sigma \Big| \leq \mathit{err}$ for all $\sigma \in ([l_0, r_0] \cap \mathbb{F}) \times \cdots \times ([l_{n-1}, r_{n-1}] \cap \mathbb{F})$,
  - where $[[e]]$ is the semantics for `float` (i.e., 64-bit double-precision format).

Notes.

- The output must be of **type `Rational`**, and the intermediate computation must be performed over rational numbers.
- For computing the square root of a rational number, use `sqrt_rational()` (although this function is slightly inexact).
- For computing the floor of log2 of a rational number, use `log2floor_rational()`.

In [ ]:
Rational = fractions.Fraction

def sqrt_rational(x: Rational) -> Rational:
    return Rational(math.sqrt(x))

def log2floor_rational(x: Rational) -> int:
    """
    Return res = floor(log2( |x| )) if x != 0, and -2^32 if x == 0.
    That is, for all x != 0, |x| = 2^res * f for some f \in [1, 2).
    """

    x = abs(x); i = 0
    if x == 0: return -2**32

    while(1):
      if 1.0 <= x < 2.0:
        break
      if x >= 2.0:
        x = x / Rational(2); i += 1
      else: # x < 1.0:
        x = x * Rational(2); i -= 1
    return i

def _ulp_bound(q: Rational) -> Rational:
    # Exact binary64 spacing at an absolute rational argument; rounded
    # arithmetic itself is never used inside this error calculation.
    q = abs(q)
    if q == 0:
        return Rational(1, 1 << 1074)
    exponent = log2floor_rational(q)
    exponent = max(exponent - 52, -1074)
    return Rational(2) ** exponent


def _sqrt_upper(q: Rational) -> Rational:
    # A rational upper bound for sqrt(q), obtained using integer arithmetic.
    # Unlike sqrt_rational(), this routine never rounds downward by accident.
    assert q >= 0
    if q == 0:
        return Rational(0)
    # Error far below the smallest subnormal spacing, with headroom.
    bits = 1100
    numer = q.numerator << (2 * bits)
    floor_scaled_square = numer // q.denominator
    scaled_root = math.isqrt(floor_scaled_square)
    bound = Rational(scaled_root + 1, 1 << bits)
    assert bound * bound > q
    return bound


def _as_rational(x: float) -> Rational:
    return Rational.from_float(x)


def get_error(e: Expr, intvs: list[tuple[float, float]]) -> Rational:
    assert isinstance(e, Expr)
    assert isinstance(intvs, list)
    assert all(isinstance(iv[0], float) and isinstance(iv[1], float)
               for iv in intvs)

    if isinstance(e, (Const, Var)):
        return Rational(0)
    if isinstance(e, Neg):
        return get_error(e.e1, intvs)

    if isinstance(e, Add):
        e1 = get_error(e.e1, intvs)
        e2 = get_error(e.e2, intvs)
        l1, r1 = (_as_rational(t) for t in get_range(e.e1, intvs))
        l2, r2 = (_as_rational(t) for t in get_range(e.e2, intvs))
        radius = max(abs(l1 + l2), abs(r1 + r2))
        return e1 + e2 + _ulp_bound(radius) / 2

    if isinstance(e, Mul):
        e1 = get_error(e.e1, intvs)
        e2 = get_error(e.e2, intvs)
        l1, r1 = (_as_rational(t) for t in get_range(e.e1, intvs))
        l2, r2 = (_as_rational(t) for t in get_range(e.e2, intvs))
        magnitude1 = max(abs(l1), abs(r1))
        magnitude2 = max(abs(l2), abs(r2))
        corners = (l1 * l2, l1 * r2, r1 * l2, r1 * r2)
        roundoff = _ulp_bound(max(map(abs, corners))) / 2
        return magnitude1 * e2 + magnitude2 * e1 + e1 * e2 + roundoff

    if isinstance(e, Sqrt):
        e1 = get_error(e.e1, intvs)
        l1, r1 = (_as_rational(t) for t in get_range(e.e1, intvs))
        assert r1 >= 0
        upper_sqrt = _sqrt_upper(r1)
        # For u,v>=0, |sqrt(u)-sqrt(v)|<=sqrt(|u-v|).
        return _sqrt_upper(e1) + _ulp_bound(upper_sqrt) / 2

    raise TypeError('Unknown expression')


In [ ]:
# Simple tests
intvs = [(-1.0, 2.0), (3.0, 4.0)]
e1 = Add( Neg(Var(1)), Const(5.0) ) # -x1+ 5.0
e2 = Mul( Var(0), Var(0) )          # x0 * x0
print( f"intvs = {intvs}" )
print( f"error({e1}) = {float(get_error(e1, intvs)).hex()}" )
print( f"error({e2}) = {float(get_error(e2, intvs)).hex()}" )

## `make_poly1()`, `make_poly2()`, and error bounds of polynomials

(1) Implement the functions `make_poly1()` and `make_poly2()`.

- They take a list `coeffs` of $n$ floats.
- They return an expression evaluating the polynomial whose coefficients are `coeffs`:
  - `make_poly1(coeffs)` must return `((coeffs[n-1] * x_0 + coeffs[n-2]) * x_0 + ...) * x_0 + coeffs[0]`.
  - `make_poly2(coeffs)` must return `((coeffs[0] + coeffs[1] * v_1) + ...) + coeffs[n-1] * v_{n-1}`, where `v_1 = x_0` and `v_k = x_0 * v_{k-1}`.

(2) Complete the following tasks using `make_poly1()`, `make_poly2()`, and `get_error()`.

- Let $p_n(x) = c_0 + c_1 x + \cdots + c_n x^n$, where $c_i$ is the float $\mathrm{round}(1/i!)$.
- For each $n \in \{2, \ldots, 5\}$, construct the two expressions for $p_n$ using `make_poly1()` and `make_poly2()`, and compute their absolute error bounds over $x_0 \in [-1, 1]$ using `get_error()`.
- Compare the two bounds for each $n$. Explain in words why one has a smaller bound than the other.

In [ ]:
def make_poly1(coeffs: list[float]) -> Expr:
    assert isinstance(coeffs, list) and len(coeffs) > 0
    assert all(isinstance(c, float) for c in coeffs)
    expr = Const(coeffs[-1])
    for c in reversed(coeffs[:-1]):
        expr = Add(Mul(expr, Var(0)), Const(c))
    return expr


def make_poly2(coeffs: list[float]) -> Expr:
    assert isinstance(coeffs, list) and len(coeffs) > 0
    assert all(isinstance(c, float) for c in coeffs)
    expr = Const(coeffs[0])
    if len(coeffs) == 1:
        return expr
    power: Expr = Var(0)
    for degree in range(1, len(coeffs)):
        expr = Add(expr, Mul(Const(coeffs[degree]), power))
        power = Mul(Var(0), power)
    return expr


In [ ]:
# Simple tests
print( make_poly1([1.0, 2.0]) )
print( make_poly2([1.0, 2.0]) )

**Answer.**

The following cell constructs both expression trees for each degree
$n\in\{2,3,4,5\}$ and computes the certified rational error bounds on
$[-1,1]$. The numerical comparison and its explanation appear in the output.


In [ ]:
intvs = [(-1.0, 1.0)]
for n in range(2, 6):
    coeffs = [float(Rational(1, math.factorial(j))) for j in range(n + 1)]
    horner = get_error(make_poly1(coeffs), intvs)
    powers = get_error(make_poly2(coeffs), intvs)
    print(f"n={n}: Horner={float(horner):.12g}, power-sum={float(powers):.12g}, "
          f"smaller={'Horner' if horner < powers else 'power-sum' if powers < horner else 'equal'}")

print("Comparison: Horner evaluation reuses one accumulator and therefore "
      "has fewer independent rounded power products. The power-sum tree "
      "recomputes powers by multiplication and accumulates their ranges "
      "and errors separately; interval dependency also widens its bounds.")
